In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 61 samples and 13399 columns.
First few rows of the data:


,os time (months),OS status,sample_title,tissue,age(0,genotype(1,tumor size (0,t stage (1,n stage (0,m stage (0,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,93.700000,0.0,TFE3-1,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,...,5.308855,3.757621,5.360897,5.656186,3.962839,4.470927,6.469223,6.748252,6.125305,5.764987
1,104.800000,0.0,TFE3-2,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 3,n0; 1:n1): 0,m0; 1:m1): 0,...,5.848443,3.730599,5.919254,6.527469,4.660299,4.747210,6.781149,6.862246,4.832972,6.616324
2,52.333333,1.0,TFE3-3,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 1,...,7.678366,5.906810,7.449775,7.975221,5.812916,6.476641,8.494724,7.960829,6.346937,7.731272
3,71.900000,0.0,TFE3-4,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,...,6.451027,4.863556,6.369385,6.755879,5.045029,6.036748,7.921962,8.600118,7.713849,7.892062
4,10.000000,1.0,TFE3-5,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 4,n0; 1:n1): 0,m0; 1:m1): 0,...,6.448179,4.869214,6.499521,7.077482,4.883677,5.596320,7.486865,7.328116,6.006584,6.856885


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for os time (months) or OS status: 7
Number of samples with complete data for both variables: 54
Number of censored samples (event_var == 0.0): 45
Number of events (event_var == 1.0): 9
Censored ratio (among complete): 45 / 54 = 0.833 (83.3%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE167573/description.json
